# C03 回测闭环与 RQAlpha 模板

本节目标：把“信号 -> 权重 -> 成本 -> 绩效”完整跑通。


## 回测闭环组件
1. 调仓日期
2. 权重函数
3. 交易成本
4. 绩效指标


In [ ]:
MODE = "offline"
START_DATE = "2021-01-01"
END_DATE = "2022-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG", "600519.XSHG"]
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


In [ ]:
def make_prices(universe, start_date, end_date):
    dates = pd.bdate_range(start_date, end_date)
    panel = {}
    for i, asset in enumerate(universe):
        ret = np.random.normal(0.0002 + i * 0.00003, 0.013, size=len(dates))
        panel[asset] = 40 * np.cumprod(1 + ret)
    return pd.DataFrame(panel, index=dates)

price = make_prices(UNIVERSE, START_DATE, END_DATE)
returns = price.pct_change().fillna(0)
price.head()


## 最小回测框架
- 月末调仓
- 过去 20 日动量 top2 等权
- 成本按换手率扣减


In [ ]:
def rebalance_dates(index):
    return pd.Series(1, index=index).resample("ME").last().index


def target_weights(price_df, as_of_date, top_n=2):
    # 动量信号：过去 20 日涨幅排名
    hist = price_df.loc[:as_of_date].tail(21)
    mom = hist.iloc[-1] / hist.iloc[0] - 1
    pick = mom.sort_values(ascending=False).head(top_n).index
    w = pd.Series(0.0, index=price_df.columns)
    w.loc[pick] = 1.0 / top_n
    return w


def simulate(price_df, fee=0.001, slip=0.0005):
    rets = price_df.pct_change().fillna(0)
    rbd = rebalance_dates(price_df.index)
    weights = pd.DataFrame(0.0, index=price_df.index, columns=price_df.columns)
    turnover = pd.Series(0.0, index=price_df.index)
    w = pd.Series(0.0, index=price_df.columns)

    for d in price_df.index:
        if d in rbd:
            tw = target_weights(price_df, d)
            turnover.loc[d] = (tw - w).abs().sum()
            w = tw
        weights.loc[d] = w

    # 用昨日权重乘今日收益，避免未来函数
    gross = (weights.shift(1).fillna(0) * rets).sum(axis=1)
    net = gross - turnover * (fee + slip)
    wealth = (1 + net).cumprod()

    return pd.DataFrame({"gross": gross, "net": net, "turnover": turnover, "wealth": wealth})

bt = simulate(price)
bt.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bt["wealth"].plot(ax=ax, title="C03 策略净值曲线")
ax.set_ylabel("NAV")
plt.show()

ann_ret = bt["net"].mean() * 252
ann_vol = bt["net"].std() * np.sqrt(252)
summary = pd.DataFrame({
    "annual_return": [ann_ret],
    "annual_vol": [ann_vol],
    "sharpe": [ann_ret / (ann_vol + 1e-12)],
    "max_drawdown": [(bt["wealth"] / bt["wealth"].cummax() - 1).min()],
    "avg_turnover": [bt["turnover"].mean()],
}).round(4)
summary


## RQAlpha 最小模板（课堂讲解）
这里保留最核心元素，便于学生迁移：
- `init`
- `handle_bar`
- `config`


In [ ]:
# from rqalpha import run_func
#
# def init(context):
#     context.pool = ["000001.XSHE", "000002.XSHE"]
#
# def handle_bar(context, bar_dict):
#     # 在此编写信号与下单逻辑
#     pass
#
# config = {
#     "base": {
#         "start_date": "2021-01-01",
#         "end_date": "2022-12-31",
#         "frequency": "1d",
#         "accounts": {"stock": 1_000_000},
#     },
#     "mod": {"sys_analyser": {"enabled": True}},
# }
#
# result = run_func(init=init, handle_bar=handle_bar, config=config)


## 小结与练习
- 成本建模是回测可信度的关键。
- 练习：改为双周调仓，比较换手率和夏普变化。
